# Notebook 2 — Sliding Window Anomaly Scanner

This is the core engine. For every sector × every calendar start day × every window size (1–60 days), it:
- Collects all historical instances of that window across all available years
- Computes Avg Return, Win Rate, Max Drawdown, Sharpe Ratio
- Saves the full raw results for Notebook 3 to filter and export

**Runtime estimate:** ~5–15 minutes depending on your machine. A progress bar will show.

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install pandas numpy tqdm openpyxl

In [14]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
# from tqdm.notebook import tqdm   # progress bar
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────

RAW_DATA_DIR    = 'data/raw'
RESULTS_DIR     = 'data'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Sliding window range
MIN_WINDOW = 1    # days
MAX_WINDOW = 60   # days

# Minimum number of historical observations needed to compute metrics
# e.g. if we have 10 years of data, a given window should appear in at least 4 years
MIN_OBSERVATIONS = 4

print(f'Window sizes: {MIN_WINDOW} → {MAX_WINDOW} days')
print(f'Min observations required per window: {MIN_OBSERVATIONS}')

Window sizes: 1 → 60 days
Min observations required per window: 4


In [3]:
# ── Load Sector Summary ───────────────────────────────────────────────────────
# Read the summary from Notebook 1 to know which sectors are available

summary_path = os.path.join('data', 'sector_summary.csv')
summary_df   = pd.read_csv(summary_path)

# Only process sectors that downloaded successfully
ok_sectors = summary_df[summary_df['Status'] == 'OK']['Sector'].tolist()

print(f'Sectors to scan: {len(ok_sectors)}')
for s in ok_sectors:
    print(f'  • {s}')

Sectors to scan: 10
  • Nifty IT
  • Nifty Bank
  • Nifty Auto
  • Nifty FMCG
  • Nifty Pharma
  • Nifty Metal
  • Nifty Realty
  • Nifty Energy
  • Nifty Financial Services
  • Nifty Media


In [4]:
# ── Helper Functions ──────────────────────────────────────────────────────────

def load_sector(sector_name):
    """
    Load a sector CSV from disk.
    Returns a DataFrame with DatetimeIndex and a 'Close' column.
    """
    filename = sector_name.replace(' ', '_').lower() + '.csv'
    filepath = os.path.join(RAW_DATA_DIR, filename)
    df = pd.read_csv(filepath, index_col='Date', parse_dates=True)
    df = df.sort_index()
    return df[['Close']]


def get_window_return(close_series, start_idx, window_size):
    """
    Given a Close price series and a starting position index,
    return the % return over the next `window_size` trading days.
    Returns None if there aren't enough days.
    """
    end_idx = start_idx + window_size
    if end_idx >= len(close_series):
        return None
    
    price_start = close_series.iloc[start_idx]
    price_end   = close_series.iloc[end_idx]
    
    if price_start == 0 or pd.isna(price_start) or pd.isna(price_end):
        return None
    
    return (price_end - price_start) / price_start * 100  # % return


def get_max_drawdown(close_series, start_idx, window_size):
    """
    Compute the maximum intra-window drawdown (%) from the entry price.
    This measures how far the price drops below entry at any point in the window.
    Returns None if data is insufficient.
    """
    end_idx = start_idx + window_size
    if end_idx >= len(close_series):
        return None
    
    window_prices = close_series.iloc[start_idx:end_idx + 1]
    entry_price   = window_prices.iloc[0]
    
    if entry_price == 0 or pd.isna(entry_price):
        return None
    
    # Drawdown = worst % drop below entry during the window
    min_price = window_prices.min()
    drawdown  = (min_price - entry_price) / entry_price * 100  # negative number
    return drawdown


def compute_metrics(returns_list, drawdowns_list):
    """
    Given a list of returns (one per historical year) and drawdowns,
    compute the 4 core metrics.
    """
    returns   = np.array([r for r in returns_list if r is not None])
    drawdowns = np.array([d for d in drawdowns_list if d is not None])
    
    if len(returns) < MIN_OBSERVATIONS:
        return None  # Not enough data points to trust
    
    avg_return   = np.mean(returns)
    win_rate     = np.sum(returns > 0) / len(returns) * 100  # % of years positive
    max_drawdown = np.min(drawdowns) if len(drawdowns) > 0 else np.nan  # worst drawdown
    
    # Sharpe: avg return / std of returns
    # (We skip risk-free rate since windows are short; relative Sharpe is what matters)
    std = np.std(returns, ddof=1)
    sharpe = avg_return / std if std > 0 else 0.0
    
    return {
        'avg_return'   : round(avg_return, 4),
        'win_rate'     : round(win_rate, 2),
        'max_drawdown' : round(max_drawdown, 4),
        'sharpe'       : round(sharpe, 4),
        'n_obs'        : len(returns),   # how many years contributed
    }


print('Helper functions defined.')

Helper functions defined.


In [5]:
# ── Core Scanning Logic ───────────────────────────────────────────────────────
#
# How the sliding window works:
#
# For a given (sector, window_size, calendar_day):
#   → Find every occurrence of that calendar day in the price history
#   → For each occurrence, measure the return over the next `window_size` trading days
#   → Collect all those returns → compute metrics
#
# Calendar day is represented as (month, day) — e.g. (10, 1) = Oct 1
# We find the nearest trading day on or after that calendar date each year.
#
# This handles weekends/holidays: if Oct 1 is a Sunday, we use Oct 2 (Monday).

def find_nearest_trading_day(df_index, year, month, day):
    """
    Given a DatetimeIndex (trading days only), find the first trading day
    on or after the target (year, month, day).
    Returns the integer position in df_index, or None if not found.
    """
    try:
        target = pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
        # Invalid date like Feb 30 — skip
        return None
    
    # Find trading days >= target within the same year
    candidates = df_index[(df_index >= target) & (df_index.year == year)]
    
    if len(candidates) == 0:
        return None
    
    nearest = candidates[0]
    return df_index.get_loc(nearest)


def scan_sector(sector_name, df):
    """
    Run the full sliding window scan for one sector.
    Returns a list of result dicts — one per (window_size, month, day) combo.
    """
    close  = df['Close']
    index  = df.index
    years  = sorted(index.year.unique())
    results = []
    
    # Iterate over every (month, day) combination in the calendar year
    # We'll use 2000 as a reference leap year to get all 366 possible dates
    all_calendar_days = pd.date_range('2000-01-01', '2000-12-31', freq='D')
    
    for window_size in range(MIN_WINDOW, MAX_WINDOW + 1):
        for cal_date in all_calendar_days:
            month = cal_date.month
            day   = cal_date.day
            
            returns_list   = []
            drawdowns_list = []
            
            for year in years:
                # Find the nearest trading day for this calendar date in this year
                pos = find_nearest_trading_day(index, year, month, day)
                if pos is None:
                    continue
                
                ret = get_window_return(close, pos, window_size)
                dd  = get_max_drawdown(close, pos, window_size)
                
                if ret is not None:
                    returns_list.append(ret)
                if dd is not None:
                    drawdowns_list.append(dd)
            
            # Compute metrics for this (window_size, month, day) combo
            metrics = compute_metrics(returns_list, drawdowns_list)
            
            if metrics is None:
                continue  # Not enough observations
            
            results.append({
                'sector'      : sector_name,
                'month'       : month,
                'day'         : day,
                'start_label' : f"{cal_date.strftime('%b')} {day:02d}",  # e.g. 'Oct 01'
                'window_days' : window_size,
                'end_label'   : (cal_date + pd.Timedelta(days=window_size)).strftime('%b %d'),
                **metrics
            })
    
    return results


print('Scan function defined.')

Scan function defined.


In [15]:
# ── Run the Scanner ───────────────────────────────────────────────────────────
# This is the main loop. Expect it to take a few minutes.

all_results = []

# for sector_name in tqdm(ok_sectors, desc='Scanning sectors'):
for sector_name in ok_sectors:
    print(f'\n→ Scanning: {sector_name}')
    
    try:
        df      = load_sector(sector_name)
        results = scan_sector(sector_name, df)
        all_results.extend(results)
        print(f'  ✓ {len(results):,} window combinations computed')
    
    except Exception as e:
        print(f'  ✗ Error scanning {sector_name}: {e}')

print(f'\n── Scan complete ──')
print(f'Total raw results: {len(all_results):,} rows')


→ Scanning: Nifty IT
  ✓ 21,900 window combinations computed

→ Scanning: Nifty Bank
  ✓ 21,900 window combinations computed

→ Scanning: Nifty Auto
  ✓ 21,900 window combinations computed

→ Scanning: Nifty FMCG
  ✓ 21,900 window combinations computed

→ Scanning: Nifty Pharma
  ✓ 21,900 window combinations computed

→ Scanning: Nifty Metal


Exception ignored while calling deallocator <function tqdm.__del__ at 0x000002331345E8D0>:
Traceback (most recent call last):
  File "d:\Projects\STOCK_ANALYZER\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\Projects\STOCK_ANALYZER\.venv\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
Exception ignored while calling deallocator <function tqdm.__del__ at 0x000002331345E8D0>:
Traceback (most recent call last):
  File "d:\Projects\STOCK_ANALYZER\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\Projects\STOCK_ANALYZER\.venv\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


  ✓ 21,900 window combinations computed

→ Scanning: Nifty Realty


Exception ignored while calling deallocator <function tqdm.__del__ at 0x000002331345E8D0>:
Traceback (most recent call last):
  File "d:\Projects\STOCK_ANALYZER\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\Projects\STOCK_ANALYZER\.venv\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


  ✓ 21,900 window combinations computed

→ Scanning: Nifty Energy
  ✓ 21,900 window combinations computed

→ Scanning: Nifty Financial Services
  ✓ 21,900 window combinations computed

→ Scanning: Nifty Media
  ✓ 21,900 window combinations computed

── Scan complete ──
Total raw results: 219,000 rows


In [16]:
# ── Convert to DataFrame & Quick Preview ─────────────────────────────────────

results_df = pd.DataFrame(all_results)

print('Columns:', list(results_df.columns))
print(f'Shape  : {results_df.shape}')
print('\nSample rows (sorted by Sharpe, descending):')
print(
    results_df
    .sort_values('sharpe', ascending=False)
    .head(10)
    .to_string(index=False)
)

Columns: ['sector', 'month', 'day', 'start_label', 'window_days', 'end_label', 'avg_return', 'win_rate', 'max_drawdown', 'sharpe', 'n_obs']
Shape  : (219000, 11)

Sample rows (sorted by Sharpe, descending):
                  sector  month  day start_label  window_days end_label  avg_return  win_rate  max_drawdown  sharpe  n_obs
Nifty Financial Services      5    9      May 09           58    Jul 06      8.8927     100.0       -7.8238  2.6359     10
              Nifty Bank      3   25      Mar 25           43    May 07      7.9783     100.0       -6.7091  2.6039     10
            Nifty Pharma      8   29      Aug 29           10    Sep 08      2.0120     100.0       -0.7228  2.5867     10
            Nifty Pharma      8   25      Aug 25           17    Sep 11      2.0690     100.0       -3.6539  2.5275     10
Nifty Financial Services      5    9      May 09           59    Jul 07      8.1811     100.0       -7.8238  2.5062     10
Nifty Financial Services      5    8      May 08       

In [17]:
# ── Basic Distribution Check ─────────────────────────────────────────────────
# Sanity check: look at the distribution of metrics across all windows.
# This helps you decide on good filter thresholds in Notebook 3.

print('═══════════════════════════════════════════')
print('       METRIC DISTRIBUTION SUMMARY')
print('═══════════════════════════════════════════')
print(results_df[['avg_return', 'win_rate', 'max_drawdown', 'sharpe']].describe().round(3).to_string())

print('\n── Win Rate distribution ──')
bins = [0, 40, 50, 60, 65, 70, 75, 80, 100]
print(pd.cut(results_df['win_rate'], bins=bins).value_counts().sort_index().to_string())

print('\n── Sharpe distribution ──')
bins = [-99, 0, 0.5, 1.0, 1.5, 2.0, 99]
print(pd.cut(results_df['sharpe'], bins=bins).value_counts().sort_index().to_string())

═══════════════════════════════════════════
       METRIC DISTRIBUTION SUMMARY
═══════════════════════════════════════════
       avg_return    win_rate  max_drawdown      sharpe
count  219000.000  219000.000    219000.000  219000.000
mean        1.950      60.401       -13.702       0.273
std         3.083      16.760         9.817       0.406
min       -13.428       0.000       -48.860      -1.852
25%         0.039      50.000       -16.687       0.008
50%         1.653      60.000       -11.146       0.274
75%         3.840      70.000        -7.179       0.519
max        14.542     100.000         0.000       2.636

── Win Rate distribution ──
win_rate
(0, 40]      36487
(40, 50]     34177
(50, 60]     50624
(60, 65]     10995
(65, 70]     38658
(70, 75]      7339
(75, 80]     23633
(80, 100]    17041

── Sharpe distribution ──
sharpe
(-99.0, 0.0]     53485
(0.0, 0.5]      107244
(0.5, 1.0]       49379
(1.0, 1.5]        7897
(1.5, 2.0]         908
(2.0, 99.0]         87


In [18]:
# ── Save Raw Results ──────────────────────────────────────────────────────────
# Save everything — Notebook 3 will apply filters and produce the final report.
# We save as a compressed CSV to keep file size manageable.

output_path = os.path.join(RESULTS_DIR, 'raw_scan_results.csv')
results_df.to_csv(output_path, index=False)

print(f'Raw results saved to: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB')
print('\n✅ Notebook 2 complete. Proceed to 03_output.ipynb')

Raw results saved to: data\raw_scan_results.csv
File size: 14.0 MB

✅ Notebook 2 complete. Proceed to 03_output.ipynb
